#### 2.3.4.3 TokenTextSplitter/CharacterTextSplitter：Split by tokens

当我们将文本拆分为块时，除了字符以外，还可以： 按Token的数量分割 （而非字符或单词数），将长文本切分成多个小块。

什么是Token？对模型而言，`Token`是文本的最小处理单位。例如（以 OpenAI 的 `cl100k_base` 分词器为例，见下面的验证）：
* 英文： "hello" → 1个Token， "ChatGPT" → 3个Token（ "Chat" + "G" + "PT" ）。
* 中文： "人工智能" → 5个Token（“智”一个字占 2 个）。换一个分词器，结果就不同。

In [1]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
for word in ["hello", "ChatGPT", "人工智能"]:
    ids = enc.encode(word)
    pieces = [enc.decode_single_token_bytes(i) for i in ids]   # 每个 token 对应的字节
    print(f"{word!r}：{len(ids)} 个 token → {pieces}")

'hello'：1 个 token → [b'hello']
'ChatGPT'：3 个 token → [b'Chat', b'G', b'PT']
'人工智能'：5 个 token → [b'\xe4\xba\xba', b'\xe5\xb7\xa5', b'\xe6\x99', b'\xba', b'\xe8\x83\xbd']


`cl100k_base` 是在 UTF-8 **字节**上做的 BPE：一个汉字在 UTF-8 里占 3 个字节，常见的字合成 1 个 token，不常见的会拆成几个。“智”被拆成 `b'\xe6\x99'` 和 `b'\xba'` 两个 token，单独哪一个都不是完整的字。后面 `TokenTextSplitter` 的乱码问题就出在这里。

为什么按Token分割？
1. 语言模型对 `输入长度的限制` 是基于Token数（如GPT-4的8k/32k Token上限），直接按字符或单词分割可能导致实际Token数超限。（确保每个文本块不超过模型的Token上限）
1. 大语言模型(LLM)通常是以token的数量作为其 计量(或收费) 的依据，所以采用token分割也有助于我们在使用时更方便的控制成本。

TokenTextSplitter 使用说明：
* 核心依据：`Token数量`。（TokenTextSplitter 严格按照 token 数量进行分割：先把整段文本编码成 token 序列，再每 chunk_size 个 token 切一刀，不看句子边界，可能在词语中间切断。）
* 优点：与LLM的Token计数逻辑一致，每块的 token 数严格不超过 chunk_size
* 缺点：不考虑语义边界；对中文还可能把一个汉字的几个 token 分到两块，解码出乱码（见下面的验证）
* 典型场景 ：需要精确控制Token数输入LLM的场景


> ⚠️ **与官方文档不同**：官方文档 Split by tokens 页的 tiktoken 一节专门提醒：中文、日文有些字符会编码成两个以上的 token，直接用 `TokenTextSplitter` 可能把同一个字符的几个 token 切到两块里，产生无效的 Unicode 字符；建议改用 `RecursiveCharacterTextSplitter.from_tiktoken_encoder` 或 `CharacterTextSplitter.from_tiktoken_encoder`。笔记原来没有这一条。

编码器说明TokenTextSplitter 底层会用到 token 编码器，后者的主要功能是将输入的文本切分为token序列，并将token序列映射为ID序列，本质上是一个 tokenizer 。


函数签名:
```python
TokenTextSplitter(
  self,
  encoding_name: str = 'gpt2',
  model_name: str | None = None,
  allowed_special: Literal['all'] | AbstractSet[str] | None = None,
  disallowed_special: Literal['all'] | Collection[str] = 'all',
  kwargs: Any = {}
)
```


签名和当前源码（`langchain-text-splitters` 1.1.2，`langchain_text_splitters/base.py`）一致，只有最后一个参数写法不同：

> ⚠️ **与当前源码不同**：`kwargs: Any = {}` 应为 `**kwargs: Any`，`chunk_size`、`chunk_overlap` 等父类参数通过它传入。

| 参数 | 默认值 | 含义 |
|---|---|---|
| `encoding_name` | `"gpt2"` | tiktoken 的编码名。默认的 `gpt2` 很旧，一般传 `cl100k_base`（GPT-4 系列）或 `o200k_base`（GPT-4o 系列） |
| `model_name` | `None` | 模型名，例如 `"gpt-4"`。传了就用 `tiktoken.encoding_for_model(model_name)` 选编码，忽略 `encoding_name` |
| `allowed_special`、`disallowed_special` | `None`（即空集合）、`"all"` | 文本里出现特殊 token（例如 GPT 的结束标记 endoftext）时怎么处理。默认全部不允许，遇到就报错；放进 `allowed_special` 的按特殊 token 编码 |
| `chunk_size`、`chunk_overlap` | 4000、200 | 父类参数，在这里单位都是 token。`chunk_overlap` 必须小于 `chunk_size`，相等也会报错（父类只要求不大于） |

`length_function`、`keep_separator`、`strip_whitespace` 对它不起作用：它不调用父类的 `_merge_splits`（2.3.3）。

**返回值**：`split_text` 返回 `list[str]`，`create_documents`、`split_documents` 返回 `list[Document]`。

**源码**：`split_text` 先把 tiktoken 编码器包装成一个 `Tokenizer`，再调用 `split_text_on_tokens`，核心是一个滑动窗口（注释是我加的）：

```python
input_ids = tokenizer.encode(text)              # 整段文本 → token id 列表
start_idx = 0
while start_idx < len(input_ids):
    cur_idx = min(start_idx + tokenizer.tokens_per_chunk, len(input_ids))
    chunk_ids = input_ids[start_idx:cur_idx]    # 取 chunk_size 个 token
    decoded = tokenizer.decode(chunk_ids)       # 解码回文本，不检查是否切在一个字的中间
    if decoded:
        splits.append(decoded)
    if cur_idx == len(input_ids):
        break
    start_idx += tokenizer.tokens_per_chunk - tokenizer.chunk_overlap   # 窗口右移，留出重叠
```

In [1]:
# 1.导入相关依赖
from langchain_text_splitters import TokenTextSplitter

# 2.初始化 TokenTextSplitter
text_splitter = TokenTextSplitter(
    chunk_size=33,  # 最大 token 数为 33
    chunk_overlap=0, # 重叠 token 数为 0
    # model_name="gpt-4", # 选择 GPT-4 模型的编码器
    encoding_name="cl100k_base",  # 使用 OpenAI 的编码器,将文本转换为 token 序列
)
# 3.定义文本
text = "人工智能是一个强大的开发框架。它支持多种语言模型和工具链。人工智能是指通过计算机程序模拟人类智能的一门科学。自20世纪50年代诞生以来，人工智能经历了多次起伏。"

# 4.开始切割
texts = text_splitter.split_text(text)

# 打印分割结果
print(f"原始文本被分割成了 {len(texts)} 个块:")
for i, chunk in enumerate(texts):
    print(f"块 {i+1}: 长度：{len(chunk)} 内容：{chunk}")
    print("-" * 50)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


原始文本被分割成了 3 个块:
块 1: 长度：29 内容：人工智能是一个强大的开发框架。它支持多种语言模型和工具链。
--------------------------------------------------
块 2: 长度：32 内容：人工智能是指通过计算机程序模拟人类智能的一门科学。自20世纪50
--------------------------------------------------
块 3: 长度：19 内容：年代诞生以来，人工智能经历了多次起伏。
--------------------------------------------------


全文 88 个 token，切成 33 + 33 + 22。第 1 块恰好在“。”处结束，只是因为第 33 个 token 正好是“。”，并不是它找了句子边界；第 2 块就在“50 | 年代”中间断开了。把 `chunk_size` 改成 32，再和官方建议的写法比较：

In [3]:
# 同一段文本，chunk_size 改成 32
print("TokenTextSplitter：")
for chunk in TokenTextSplitter(encoding_name="cl100k_base", chunk_size=32, chunk_overlap=0).split_text(text):
    print(f"  {len(enc.encode(chunk)):>2} token | {chunk!r}")

# 官方建议：先按分隔符切，再按 token 数合并
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=32,
    chunk_overlap=0,
    separators=["\n\n", "\n", "。", "！", "？", "；", "，", "、", " ", ""],   # 2.3.4.2 的中文分隔符
    keep_separator="end",
)
print("RecursiveCharacterTextSplitter.from_tiktoken_encoder：")
for chunk in splitter.split_text(text):
    print(f"  {len(enc.encode(chunk)):>2} token | {chunk!r}")

TokenTextSplitter：
  32 token | '人工智能是一个强大的开发框架。它支持多种语言模型和工具链'
  32 token | '。人工智能是指通过计算机程序模拟人类智能的一门科学。自20世�'
  24 token | '�50年代诞生以来，人工智能经历了多次起伏。'
RecursiveCharacterTextSplitter.from_tiktoken_encoder：
  18 token | '人工智能是一个强大的开发框架。'
  15 token | '它支持多种语言模型和工具链。'
  26 token | '人工智能是指通过计算机程序模拟人类智能的一门科学。'
  29 token | '自20世纪50年代诞生以来，人工智能经历了多次起伏。'


* `TokenTextSplitter`：第 2 块末尾和第 3 块开头各有一个“�”。“纪”在 `cl100k_base` 里占 2 个 token，正好被切到了两块里，单独解码哪一半都不是合法的字，tiktoken 就用替换字符“�”（U+FFFD）代替。这两块存进向量库后，“纪”字就丢了。第 1 块的“。”也被分到了第 2 块开头。
* `RecursiveCharacterTextSplitter.from_tiktoken_encoder`：先按标点切，再按 token 数合并，每块都是完整的句子，也都不超过 32 个 token。切到单个字符为止，所以不会把一个字拆开。

In [2]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter
import tiktoken  # 用于计算Token数量


# 2.定义通过Token切割器
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", # 使用 OpenAI 的编码器
    chunk_size=18,
    chunk_overlap=0,
    separator="。",  # 指定中文句号为分隔符
    keep_separator=False,  # chunk中是否保留分隔符
)
# 3.定义文本
text = "人工智能是一个强大的开发框架。它支持多种语言模型和工具链。今天天气很好，想出去踏青。但是又比较懒不想出去，怎么办"

# 4.开始切割
texts = text_splitter.split_text(text)

print(f"原始文本被分割成了 {len(texts)} 个块:")
for i, chunk in enumerate(texts):
    print(f"块 {i+1}: 长度：{len(chunk)} 内容：{chunk}")
    print("-" * 50)

原始文本被分割成了 4 个块:
块 1: 长度：14 内容：人工智能是一个强大的开发框架
--------------------------------------------------
块 2: 长度：13 内容：它支持多种语言模型和工具链
--------------------------------------------------
块 3: 长度：12 内容：今天天气很好，想出去踏青
--------------------------------------------------
块 4: 长度：14 内容：但是又比较懒不想出去，怎么办
--------------------------------------------------


这里只有 `length_function` 换成了数 token，切分规则还是 2.3.4.1 的 `CharacterTextSplitter`：

* 按“。”切出 4 个片段，分别是 17、14、18、21 个 token。任意两个加起来都超过 18，所以每句单独一块；`keep_separator=False`，块末尾的“。”都丢了；
* 第 4 块有 21 个 token，超过了 `chunk_size=18`：`CharacterTextSplitter` 不会在片段内部再切，而且它是最后一块，连警告都没有（2.3.4.1 的补充）。官方文档也提醒过：用 `CharacterTextSplitter.from_tiktoken_encoder` 切出的块可能超过 `chunk_size`；要严格限制 token 数，用上面的 `RecursiveCharacterTextSplitter.from_tiktoken_encoder`。

#### 2.3.4.4 SemanticChunker：语义分块
SemanticChunking（语义分块）是一种更高级的文本分割方法（LangChain 的 `SemanticChunker` 在已经停止维护的 `langchain-experimental` 包里，见 2.3.2），它超越了传统的基于字符或固定大小的分块方式，而是根据 文本的语义结构 进行智能分块，使每个分块保持 语义完整性 ，从而提高检索增强生成(RAG)等应用的效果。

* 语义分块与传统分割方式的区别

通过将文本转化为 向量（Embedding） ，去计算前后句子的语义差异，当发现前后两句话的语义变化很大（超过设定的阈值）时，就在这里一刀切断。这样能保证切分出来的每一个文本块（Chunk）在含义上是完整、连贯的。

由于langchain的包已经停止维护，这里我们使用其他还算主流的方案
：chonkie（本项目装的是 1.7.0）的 `SemanticChunker`。它和 LangChain 的同名类没有关系，返回的也不是 LangChain 的 `Document`（见下面的返回值）。

In [ ]:
from chonkie import SemanticChunker

# Basic initialization with default parameters
# 已经下载过了,不用再运行
chunker = SemanticChunker(
    embedding_model="minishlab/potion-base-32M",  # Default model
    threshold=0.8,                               # Similarity threshold (0-1)
    chunk_size=2048,                             # Maximum tokens per chunk
    similarity_window=3,                         # Window for similarity calculation
    skip_window=0                                # Skip-and-merge window (0=disabled)
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

In [3]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain.embeddings import init_embeddings
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# 加载文本
with open("load/09-ai1.txt", encoding="utf-8") as f:
    text = f.read()  #返回字符串

from chonkie import SemanticChunker
from chonkie.embeddings import Model2VecEmbeddings

chunker = SemanticChunker(
    embedding_model=Model2VecEmbeddings("huggingface/minishlab/potion-base-32M"),
    threshold=0.8,
    chunk_size=2048,
)

chunks = chunker.chunk(text)
# Process chunks
for chunk in chunks:
    print(f"Chunk: {chunk.text[:50]}...")
    print(f"Tokens: {chunk.token_count}")

Chunk: 人工智能综述：发展、应用与未来展望

摘要
人工智能（Artificial Intelligence...
Tokens: 634
Chunk: 2012年，深度学习在图像识别领域取得了突破性进展，标志着人工智能进入了一个新的阶段。深度学习通过多...
Tokens: 166
Chunk: 机器学习是人工智能的核心技术之一，通过算法使计算机从数据中学习并做出决策。常见的机器学习算法包括监督...
Tokens: 355
Chunk: 计算机视觉是人工智能的另一个重要分支，致力于使计算机能够理解和处理图像和视频。计算机视觉技术广泛应用...
Tokens: 98
Chunk: 人工智能的应用领域
4.1 医疗健康
人工智能在医疗健康领域的应用包括疾病诊断、药物研发、个性化医疗...
Tokens: 314
Chunk: 人工智能的挑战与机遇
5.1 挑战
...
Tokens: 15
Chunk: 人工智能发展过程中面临的主要挑战包括数据隐私、算法偏见、安全性问题等。数据隐私问题涉及到个人数据的收...
Tokens: 513


**chonkie `SemanticChunker` 的切分过程**（据 `chonkie/chunker/semantic.py` 源码）：

1. 断句：按 `delim` 把文本切成“句子”，太短的句子（短于 `min_characters_per_sentence`）并到相邻的句子里；
2. 算相似度：从第 `similarity_window + 1` 句开始，把每一句和它前面 `similarity_window` 句拼成的窗口分别算向量，求余弦相似度，得到一条“相似度曲线”。相似度低，说明这一句和前文的话题不一样；
3. 找切分点：先用 Savitzky-Golay 滤波把曲线平滑，找出曲线的局部最低点，再按 `threshold` 从这些最低点里留下相似度较低的一部分，相邻两个切分点至少隔 `min_sentences_per_chunk` 句；
4. （可选）`skip_window > 0` 时，把后面几组里相似度够高的组合并过来；
5. 超过 `chunk_size` 个 token 的组，按句子拆小；
6. 每组生成一个 `Chunk`。

文本太短（句子数不超过 `similarity_window`）时不切分，整段作为一块返回。

**参数**

| 参数 | 默认值 | 含义 |
|---|---|---|
| `embedding_model` | `"minishlab/potion-base-32M"` | 算句子向量的模型。可以传字符串（Hugging Face 模型名、本地路径，或 `"openai://text-embedding-3-small"` 这样的写法），也可以传 chonkie 的嵌入对象（`chonkie.embeddings` 里的 `Model2VecEmbeddings`、`SentenceTransformerEmbeddings`、`OpenAIEmbeddings` 等）。**不能直接传 LangChain 的嵌入类**：构造函数只接受这两种类型，否则报 `ValueError`。块的 token 数也用这个模型的分词器计算 |
| `threshold` | `0.8` | 取值在 0 和 1 之间。挑切分点时，从相似度曲线的局部最低点里，大约留下相似度最低的 `threshold` 比例（0.8 约等于留下最低的 80%）。**越大，切分点越多，块越小**。`skip_window > 0` 时，它还被当作余弦相似度的下限，决定两组能不能合并 |
| `chunk_size` | `2048` | 每块最多多少 token（按 `embedding_model` 的分词器计）。超过的组按句子拆小；单个句子超过它，不会再切 |
| `similarity_window` | `3` | 每一句和前面几句组成的窗口比较相似度 |
| `min_sentences_per_chunk` | `1` | 相邻两个切分点之间至少隔几句 |
| `min_characters_per_sentence` | `24` | 句子的最短长度，短于它的并到相邻句子里。底层按 UTF-8 **字节**计，一个汉字占 3 个字节，所以对中文大约是 8 个字 |
| `delim` | `[". ", "! ", "? ", "\n"]` | 断句用的分隔符。默认都是英文标点加空格，**没有中文的“。”**，中文文本只会按换行断句 |
| `include_delim` | `"prev"` | 分隔符留在前一句的末尾（`"prev"`）还是后一句的开头（`"next"`） |
| `skip_window` | `0` | 0 表示关闭。大于 0 时，每组会和后面 `skip_window` 组比较，相似度不低于 `threshold` 就把它们（连同中间的组）合成一组，把被打断的同一话题接回去 |
| `filter_window`、`filter_polyorder`、`filter_tolerance` | `5`、`3`、`0.2` | Savitzky-Golay 滤波的窗口长度、多项式阶数，以及判断局部最低点的容差。用来平滑相似度曲线，一般不用改 |

> ⚠️ **与 chonkie 文档不同**：文档和代码注释都把 `threshold` 写成“相似度阈值（Similarity threshold）”，读起来像“相似度低于 0.8 就切”。但在 1.7.0 的源码里，挑切分点时 `threshold` 被传给 `chonkie_core.filter_split_indices`，这个函数的说明是 “Percentile threshold (0.0-1.0)”：它在候选的局部最低点之间按排名挑选，和相似度的绝对值无关。文档里“值越小，组越大”的说法和这个行为是一致的。

**返回值**：`chunk(text)` 返回 `list[Chunk]`。`Chunk` 是 chonkie 自己的类型，不是 LangChain 的 `Document`，主要字段：

| 字段 | 含义 |
|---|---|
| `text` | 块的文本 |
| `start_index`、`end_index` | 块在原文中的起止字符位置 |
| `token_count` | 块的 token 数（按 `embedding_model` 的分词器计） |
| `id` | 自动生成，以 `chnk_` 开头 |
| `embedding`、`context`、`metadata` | 默认是空的，chonkie 的其他组件（如 refinery）会填写 |

块之间没有重叠，所有块按顺序拼起来正好是原文。`chunk_batch(texts)` 可以一次处理多段文本，返回 `list[list[Chunk]]`。

对照上面的输出：

* `09-ai1.txt` 是中文，默认的 `delim` 只能按换行断句，2243 个字只分出 25 个“句子”，基本是一行一句，最后得到 7 块；
* 第 6 块只有“人工智能的挑战与机遇\n5.1 挑战\n”两行标题（15 个 token）。语义切分不控制块的大小，可能切出只有标题的块（2.3.2 说的“长度极不均匀”）。切分点还常落在小标题之后，例如第 1 块以小标题“2.3 深度学习的突破”结尾，而这个标题讲的内容在第 2 块；
* token 数是用 potion-base-32M 的分词器算的（它来自 bge-base-en-v1.5），中文基本是一个字一个 token。

中文还有一个更根本的问题：potion-base-32M 是英文模型。它的模型卡（本地 `huggingface/minishlab/potion-base-32M/README.md`）写着：从 `baai/bge-base-en-v1.5` 蒸馏而来，用英文语料 C4 训练。下面比较它对中文、英文句子的相似度：

In [5]:
from chonkie.embeddings import Model2VecEmbeddings

embeddings = Model2VecEmbeddings("huggingface/minishlab/potion-base-32M")

sentences = {
    "中文": ["人工智能在医疗领域应用广泛。", "深度学习推动了人工智能的发展。", "今天中午吃了牛肉面，很好吃。"],
    "英文": ["AI is widely used in healthcare.", "Deep learning drives the progress of AI.", "I had beef noodles for lunch today."],
}
for lang, (a, b, c) in sentences.items():
    va, vb, vc = embeddings.embed_batch([a, b, c])
    print(f"{lang}：相关的两句 {embeddings.similarity(va, vb):.2f}，无关的两句 {embeddings.similarity(va, vc):.2f}")

中文：相关的两句 0.75，无关的两句 0.58
英文：相关的两句 0.27，无关的两句 -0.02


英文里相关的两句 0.27、无关的两句 -0.02，区分很明显；中文里无关的两句也有 0.58，和相关的 0.75 差得不多。模型分不清中文的话题，相似度曲线的高低起伏，很大一部分不是话题变化造成的。

所以中文项目用语义切分，要改两处：

1. 换成支持中文的嵌入模型。例如同一系列的 `minishlab/potion-multilingual-128M`（Hugging Face 上标注支持中文），或者后面学向量化时用的中文嵌入模型；
2. 把 `delim` 改成中文标点。

另外，`Chunk` 要转成 LangChain 的 `Document`，才能交给 LangChain 的向量库。下面演示后两点（模型仍用 potion-base-32M，只看用法）：

In [6]:
from langchain_core.documents import Document

chunker = SemanticChunker(
    embedding_model=embeddings,          # 演示仍用 potion-base-32M；中文项目应换成支持中文的模型
    threshold=0.8,
    chunk_size=2048,
    delim=["。", "！", "？", "\n"],       # 中文句末标点 + 换行
)
chunks = chunker.chunk(text)
print(f"切成 {len(chunks)} 块，每块的 token 数：{[c.token_count for c in chunks]}")

# chonkie 的 Chunk → LangChain 的 Document
docs = [
    Document(
        page_content=c.text,
        metadata={"source": "load/09-ai1.txt", "start_index": c.start_index, "token_count": c.token_count},
    )
    for c in chunks
]
print(docs[1].metadata, repr(docs[1].page_content[:30]))

切成 23 块，每块的 token 数：[185, 36, 154, 18, 119, 85, 27, 91, 67, 18, 158, 32, 53, 67, 114, 27, 204, 74, 36, 146, 214, 38, 132]
{'source': 'load/09-ai1.txt', 'start_index': 210, 'token_count': 36} '最后，本文对人工智能的未来发展进行了展望，提出了可能的突破方'


按中文标点断句后，块从 7 个变成了 23 个，其中不少块只有十几个 token。句子分对了，但用的还是英文模型，相似度本身不可靠，所以这个结果也不能说明切得更好。中文项目两处都要改。

#### 2.3.4.5 HTMLHeaderTextSplitter

HTMLHeaderTextSplitter是一种专门用于处理HTML文档的文本分割方法，它根据HTML的 标题标签（如`<h1>`、`<h2>`等） 将文档划分为逻辑分块，同时保留标题的层级结构信息。

```python
HTMLHeaderTextSplitter(
  self,
  headers_to_split_on: list[tuple[str, str]],
  return_each_element: bool = False
)
```


签名和当前源码（`langchain-text-splitters` 1.1.2，`html.py`）一致。它**不继承** `TextSplitter`（2.3.3），没有 `chunk_size`、`chunk_overlap`。

| 参数 | 默认值 | 含义 |
|---|---|---|
| `headers_to_split_on` | 必填 | `(标签, 名称)` 的列表，例如 `("h1", "标题1")`：遇到这些标签就开始新的一段，标签里的文字以“名称”为键写进 metadata。源码会按 h1、h2… 的级别排好序 |
| `return_each_element` | `False` | `False`：同一组标题下连续的内容合成一个 `Document`；`True`：每个 HTML 元素（段落、列表项等）各生成一个 `Document` |

**返回值**：`split_text(html)` 返回 `list[Document]`。

* 每个标题本身也会单独生成一个 `Document`（源码注释：Always yield a Document for the header），所以上面的输出里，每个标题各占一行；
* 内容的 metadata 是它所在的各级标题；
* 一个标题都没找到时，整篇作为一个 `Document` 返回。

读本地文件用 `split_text_from_file(路径或文件对象)`。

> ⚠️ **与官方文档不同**：官方文档还在用 `split_text_from_url(url)` 直接切网页，但当前源码已把它标为弃用（since 1.1.2，2.0.0 删除），提示改成自己请求网页、再把 HTML 传给 `split_text`。

In [5]:
# 1.导入相关依赖
from langchain_text_splitters import HTMLHeaderTextSplitter

# 2.定义HTML文件
html_string = """
<!DOCTYPE html>
<html>
<body>
    <div>
        <h1>欢迎来到尚硅谷！</h1>
        <p>尚硅谷是专门培训IT技术方向</p>
        <div>
            <h2>尚硅谷老师简介</h2>
            <p>尚硅谷老师拥有多年教学经验，都是从一线互联网下来</p>
            <h3>尚硅谷北京校区</h3>
            <p>北京校区位于宏福科技园区</p>
        </div>
    </div>
</body>
</html>
"""

# 4.用于指定要根据哪些HTML标签来分割文本
headers_to_split_on = [
    ("h1", "标题1"),
    ("h2", "标题2"),
    ("h3", "标题3"),
]

# 5.定义HTMLHeaderTextSplitter分割器
html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# 6.分割器分割
html_header_splits = html_splitter.split_text(html_string)

for html_splits in html_header_splits:
    print(html_splits)

page_content='欢迎来到尚硅谷！' metadata={'标题1': '欢迎来到尚硅谷！'}
page_content='尚硅谷是专门培训IT技术方向' metadata={'标题1': '欢迎来到尚硅谷！'}
page_content='尚硅谷老师简介' metadata={'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介'}
page_content='尚硅谷老师拥有多年教学经验，都是从一线互联网下来' metadata={'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介'}
page_content='尚硅谷北京校区' metadata={'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介', '标题3': '尚硅谷北京校区'}
page_content='北京校区位于宏福科技园区' metadata={'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介', '标题3': '尚硅谷北京校区'}


它不限制块的大小，一个标题下的内容再长也是一个 `Document`。官方文档的做法（How to constrain chunk sizes）是再用 `RecursiveCharacterTextSplitter.split_documents` 切一次，metadata 会复制到每一块上：

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=15,
    chunk_overlap=0,
    separators=["\n\n", "\n", "。", "！", "？", "；", "，", "、", " ", ""],
    keep_separator="end",
)
for doc in text_splitter.split_documents(html_header_splits):
    print(repr(doc.page_content), doc.metadata)

'欢迎来到尚硅谷！' {'标题1': '欢迎来到尚硅谷！'}
'尚硅谷是专门培训IT技术方向' {'标题1': '欢迎来到尚硅谷！'}
'尚硅谷老师简介' {'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介'}
'尚硅谷老师拥有多年教学经验，' {'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介'}
'都是从一线互联网下来' {'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介'}
'尚硅谷北京校区' {'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介', '标题3': '尚硅谷北京校区'}
'北京校区位于宏福科技园区' {'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介', '标题3': '尚硅谷北京校区'}


`return_each_element=False`（默认）时还有一个坑：标题被 `<div>` 包着时，div 后面的内容会让前面内容的 metadata 出错：

In [9]:
nested_html = """
<html><body>
    <h1>欢迎来到尚硅谷！</h1>
    <div>
        <h2>尚硅谷老师简介</h2>
        <p>尚硅谷老师拥有多年教学经验</p>
    </div>
    <p>这一段在 div 外面，只属于标题1</p>
</body></html>
"""
for each in [False, True]:
    print(f"return_each_element={each}：")
    splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on, return_each_element=each)
    for doc in splitter.split_text(nested_html):
        print("  ", repr(doc.page_content), doc.metadata)

return_each_element=False：
   '欢迎来到尚硅谷！' {'标题1': '欢迎来到尚硅谷！'}
   '尚硅谷老师简介' {'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介'}
   '尚硅谷老师拥有多年教学经验  \n这一段在 div 外面，只属于标题1' {'标题1': '欢迎来到尚硅谷！'}
return_each_element=True：
   '欢迎来到尚硅谷！' {'标题1': '欢迎来到尚硅谷！'}
   '尚硅谷老师简介' {'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介'}
   '尚硅谷老师拥有多年教学经验' {'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介'}
   '这一段在 div 外面，只属于标题1' {'标题1': '欢迎来到尚硅谷！'}


* `False`：“尚硅谷老师拥有多年教学经验”和 div 外面那段合成了一块，metadata 里没有“标题2”。源码里，块只在遇到下一个标题时才结束；h2 的作用范围到 div 结束为止，处理到 div 外面那段时，源码把“标题2”从当前标题里删掉了，却没有先结束正在攒的块。块结束时取的是那一刻的标题，于是前一段也丢了“标题2”；
* `True`：每个元素单独生成 `Document`，各自的标题都对。

真实网页的标题常被 `<div>` 包着。要依赖标题 metadata 做检索或标注出处时，先检查输出，或者改用 `return_each_element=True`。